<a href="https://colab.research.google.com/github/DaviAlves06/ai-pharmacy-agent/blob/main/Agente_Farmac%C3%AAutico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# PROJETO PRÁTICO — Agente Farmacêutico Explicador de Bulas (RAG)


Este projeto incorpora:
- Múltiplos documentos
- Chunking mais estratégico
- Enriquecimento com metadados
- Maior controle semântico e rastreabilidade



## Dependências

```bash
!pip install -qU \
  "langchain>=0.2.0,<0.4.0" \
  langchain-community \
  langchain-google-genai \
  chromadb \
  pypdf \
  -qU sentence-transformers
```


In [ ]:
!pip install -qU \
  "langchain>=0.2.0,<0.4.0" \
  langchain-community \
  langchain-google-genai \
  chromadb \
  pypdf \
  -qU sentence-transformers

In [ ]:
# Importa o módulo para manipulação de variáveis de ambiente
import os

# Loader responsável por ler arquivos PDF
from langchain_community.document_loaders import PyPDFLoader

# Responsável por dividir textos grandes em chunks menores
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Embeddings com Gemini (Google Generative AI)
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Banco vetorial para armazenamento e busca semântica
from langchain_community.vectorstores import Chroma

# Modelo de linguagem conversacional (Gemini)
from langchain_google_genai import ChatGoogleGenerativeAI

# Cadeia pronta de Perguntas e Respostas com RAG
from langchain.chains import RetrievalQA

In [ ]:
import os
import getpass

# Defina sua chave da Google AI (Gemini) como variável de ambiente
# Gere a chave em: https://ai.google.dev/gemini-api/docs/api-key
if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Informe sua GOOGLE_API_KEY: ")


## 1️⃣ Definição do Problema

Bulas farmacêuticas possuem:
- Linguagem técnica
- Grande volume de texto
- Informações sensíveis

O objetivo do agente é **interpretar corretamente as bulas**,
respondendo perguntas **somente com base nos documentos**,
evitando qualquer tipo de alucinação.



## 2️⃣ Seleção e Organização das Bulas




In [ ]:

# Lista com os caminhos dos arquivos PDF das bulas
caminhos_bulas = [
    "dipirona.pdf",
    "paracetamol.pdf"
]

# Lista que armazenará todos os documentos carregados
documentos = []

# Percorre cada bula
for caminho in caminhos_bulas:

    # Cria o loader do PDF
    loader = PyPDFLoader(caminho)

    # Carrega o conteúdo do PDF
    docs = loader.load()

    # Adiciona o nome do medicamento como metadado
    for doc in docs:
        doc.metadata["medicamento"] = caminho.split("/")[-1].replace(".pdf", "")

    # Adiciona os documentos à lista principal
    documentos.extend(docs)

# Exibe a quantidade total de páginas carregadas
len(documentos)


In [ ]:
print(documentos)


## 3️⃣ Extração, Limpeza e Chunking



In [ ]:

# Cria o objeto responsável pelo chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,      # Tamanho máximo de cada chunk
    chunk_overlap=150   # Sobreposição entre chunks
)

# Divide os documentos em chunks
chunks = text_splitter.split_documents(documentos)

# Quantidade total de chunks gerados
len(chunks)


In [ ]:
chunks


## 4️⃣ Enriquecimento com Metadados


In [ ]:

# Percorre cada chunk para classificar semanticamente seu conteúdo
for chunk in chunks:

    # Normaliza o texto para facilitar as verificações
    texto = chunk.page_content.lower()

    # Identificação do medicamento
    if "identificação do medicamento" in texto or "composição" in texto:
        chunk.metadata["categoria"] = "identificacao"

    # Indicações terapêuticas
    elif "indicação" in texto or "para que este medicamento é indicado" in texto:
        chunk.metadata["categoria"] = "indicacao"

    # Funcionamento do medicamento
    elif "como este medicamento funciona" in texto or "ação" in texto:
        chunk.metadata["categoria"] = "como_funciona"

    # Contraindicações
    elif "contraindicação" in texto or "quando não devo usar" in texto:
        chunk.metadata["categoria"] = "contraindicacao"

    # Advertências e precauções
    elif "advertência" in texto or "precaução" in texto or "o que devo saber antes de usar" in texto:
        chunk.metadata["categoria"] = "advertencias_precaucoes"

    # Interações medicamentosas
    elif "interação" in texto or "interações medicamentosas" in texto:
        chunk.metadata["categoria"] = "interacoes"

    # Posologia e modo de uso
    elif "dose" in texto or "posologia" in texto or "como devo usar" in texto:
        chunk.metadata["categoria"] = "posologia_modo_uso"

    # Reações adversas
    elif "reações adversas" in texto or "quais os males" in texto:
        chunk.metadata["categoria"] = "reacoes_adversas"

    # Armazenamento
    elif "onde, como e por quanto tempo posso guardar" in texto or "armazenar" in texto:
        chunk.metadata["categoria"] = "armazenamento"

    # Superdosagem
    elif "quantidade maior do que a indicada" in texto or "superdosagem" in texto:
        chunk.metadata["categoria"] = "superdosagem"

    # Conteúdo geral / administrativo
    else:
        chunk.metadata["categoria"] = "geral"



In [ ]:
import random

# Seleciona dois chunks aleatórios
chunks_aleatorios = random.sample(chunks, 2)

# Imprime os metadados e um trecho do conteúdo
for i, chunk in enumerate(chunks_aleatorios, start=1):
    print(f"\n--- Chunk Aleatório {i} ---")
    print("Metadados:")
    print(chunk.metadata)
    print("\nConteúdo (início):")
    print(chunk.page_content[:300])



## 5️⃣ Geração de Embeddings e Banco Vetorial


In [ ]:
#!pip install chromadb


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Embeddings locais (HuggingFace) – estáveis, gratuitos e sem depender de API externa
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Cria o banco vetorial com os chunks
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_bulas"
)


## 6️⃣ Recuperação de Contexto (Retriever)


In [ ]:

# Cria o retriever para busca semântica
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}  # Número de chunks retornados
)



## 7️⃣ Integração com Pipeline RAG


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.1,   # respostas mais factuais
    transport="rest",  # evita gRPC e o erro 'Illegal metadata'
    timeout=60,        # tempo máx. de cada chamada (segundos)
    max_retries=1,     # no máx. 1 retry rápido; evita ficar 10+ minutos
)

# Cria a cadeia RAG
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)


## 8️⃣ Testes e Validação das Respostas


In [ ]:
# Pergunta de teste
pergunta = "Quais são as contraindicações da dipirona?"

# Executa a pergunta no agente RAG (forma atual)
resposta = qa_chain.invoke(pergunta)

print("Pergunta:")
print(pergunta)

print("\nResposta do Agente:")
print(resposta["result"])

print("\nTrechos utilizados como contexto:\n")

# Percorre os documentos recuperados
for i, doc in enumerate(resposta["source_documents"], start=1):
    print(f"--- Trecho {i} ---")

    # Conteúdo recuperado
    print("\nConteúdo do chunk:")
    print(doc.page_content)
    print("\n")


In [ ]:
# Pergunta de teste 1
pergunta = "Quais são as recomendações do paracetamol?"

# Executa a pergunta no agente RAG
resposta = qa_chain.invoke(pergunta)

print("Pergunta:")
print(pergunta)

print("\nResposta do Agente:")
print(resposta["result"])

print("\nTrechos utilizados como contexto:\n")

for i, doc in enumerate(resposta["source_documents"], start=1):
    print(f"--- Trecho {i} ---")
    print(f"Medicamento: {doc.metadata.get('medicamento', 'N/A')}")
    print(f"Categoria: {doc.metadata.get('categoria', 'N/A')}")
    print(f"Documento: {doc.metadata.get('source', 'Documento desconhecido')}")
    print(f"Página: {doc.metadata.get('page', 'N/A')}")
    print("\nConteúdo do chunk:")
    print(doc.page_content)
    print("\n")
